In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts
from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
%matplotlib inline

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42


## Model definition

Helper functions, the CNN architecture, the custom rollout-training `RolloutEmulator` model,
and the physical-attractor checkpoint callback that only saves weights when the C₂ constraint
and lag-1 ACF causality check both pass.

In [ ]:
def set_seeds(seed=seed):
    # set random see for tensorflow, keras, numpy etc.
    tf.keras.utils.set_random_seed(seed)
    # only relevant if calculation is done on GPU
    tf.config.experimental.enable_op_determinism()

def create_windows(data, window_size, rollout_length):
    X_list, y_list, Insol_rollout_list = [], [], []
    for i in range(len(data) - window_size - rollout_length):
        X_list.append(data[i:i + window_size,:])
        y_list.append(data[i + window_size: i + window_size + rollout_length, :-1])
        Insol_rollout_list.append(data[i + window_size: i + window_size + rollout_length, -1])

    return np.array(X_list), np.array(y_list), np.array(Insol_rollout_list)

def CNN_model(window_size, n_features, conv_filter = 12, conv_kernel = 4, dense_unit = 12, rate = 0.2):
    """
    Convolutional Neural Network architecture to be used for the SR-LPNE model
    --------------------------------------------------------------------------

    window_size - input window size
    n_features - number of input variables (here set to 4 for X,Y,Z,Insol

    """
    # 0 parameters
    inp = layers.Input(shape=(window_size, n_features))

    x = layers.GaussianNoise(0.05)(inp) #increased from 0.05

    # 16 filters and 5 kernel size
    # size = K*4K*F+F=336 params
    x = layers.Conv1D(filters = conv_filter,
                      kernel_size = conv_kernel, # changed from 5 to 50 for high resolution
                      activation = 'swish',
                      padding = "causal")(x)

    x = layers.LayerNormalization()(x)

    # extract last tensor from convolutional layer output
    # size = 2F = 2*16 = 32 params
    x = layers.Lambda(lambda x: x[:, -1, :])(x)

    # size = F*U+U = 16*32+32 = 544 params
    x = layers.Dense(units = dense_unit,
                     activation="swish")(x)

    # dropout layer to reduce overfitting; dropout rate is a hyperparameter
    x = layers.Dropout(rate=rate)(x)

    # output layer, returns the vector [\Delta_X,\Delta_Y,\Delta_Z]
    # size = 3*U+3 = 3*32+3 = 99 params
    out = layers.Dense(units = 3,
                       activation = 'linear',
                       kernel_initializer = 'zeros',
                       bias_initializer = 'zeros',
                       use_bias=True)(x)

    # total size = 1011 params

    # physical constraint: scaling by maximum incerement in data
    # WARNING: NEEDS tO BE ADJUSTED FOR DIFFERENT DATA SETS
    #out = out * tf.constant([0.47893476,0.78733971,0.21167531], dtype=tf.float32)

    return Model(inputs = inp,outputs = out)

class RolloutEmulator(tf.keras.Model):
    """
    custom keras model class for rollout training, validation and prediction.

    loss_type selects between two loss functions, both computed on the same
    rollout trajectory (true_path vs. p_stack):
      - "sobolev" (default): pointwise MSE + jacobian + hessian loss, weighted
        by L_2_weight / J_weight / H_weight (J_weight/H_weight default to 0,
        i.e. plain MSE, unless you pass them explicitly).
      - "dwt": Harmonized MSE (hMSE) on a multi-level Haar DWT of the
        trajectory, per "The Procrustean Bed of Time Series" (arXiv:2512.18610),
        Eq. 21-23. Ported from SR-LPNE_DWT.ipynb - see that notebook for the
        derivation/commentary on the DWT machinery below.

    The rollout loop itself is identical either way (it always collects the
    per-step jacobian, which "sobolev" mode uses and "dwt" mode just ignores) -
    only the loss computed from the finished trajectory differs.
    """
    def __init__(self,brain,rollout_length,vali_rollout_length,window_size,n_features,
                 loss_type="sobolev",
                 L_2_weight=1.0,J_weight=0,H_weight=0,
                 dwt_gamma=0.5,dwt_beta=0.3,dwt_eps=1e-6,dwt_levels=2):
        """
        initializer method for the rollout trainig class
        ------------------------------------------------

        brain               - model to train via rollout training
        rollout_length      - training rollout length
        vali_rollout_length - validation rollout length
        loss_type            - "sobolev" or "dwt" (default "sobolev")
        L_2_weight          - L_2 term weight, used by both loss types (default 1.0)
        J_weight             - jacobian term weight for Sobolev loss (default 0)
        H_weight             - hessian term weight for Sobolev loss (default 0)
        dwt_gamma            - Eq. 21 gamma for the DWT loss (paper default 0.5); unused if loss_type="sobolev"
        dwt_beta             - Eq. 23 beta: EMA smoothing factor for the DWT loss (paper default 0.3); unused if loss_type="sobolev"
        dwt_eps              - Eq. 21 epsilon: minimum threshold preventing weight explosion; unused if loss_type="sobolev"
        dwt_levels            - number of recursive DWT decomposition levels (default 2); unused if loss_type="sobolev"
        noise_level         - standard deviation of normally distributed noise used for noise injection during training
        """

        super(RolloutEmulator, self).__init__()
        assert loss_type in ("sobolev", "dwt"), f"loss_type must be 'sobolev' or 'dwt', got {loss_type!r}"
        self.loss_type = loss_type

        self.brain = brain
        self.rollout_length = rollout_length
        self.vali_rollout_length = vali_rollout_length
        self.L_2_weight = L_2_weight
        self.jacobian_weight = J_weight
        self.hessian_weight = H_weight
        self.noise_level = 1e-2 # change to zero at some point
        self.best_abs = 1e6
        self.brain.build((None, window_size, n_features))

        if self.loss_type == "dwt":
            # --- Harmonized MSE (hMSE) setup, ported from SR-LPNE_DWT.ipynb -----
            # Haar wavelet: orthonormal 2-tap low-pass (approximation) / high-pass
            # (detail) filters, shaped (filter_width, in_channels=1, out_channels=1)
            # for conv1d.
            self.dwt_low  = tf.constant([[[1/np.sqrt(2)]], [[ 1/np.sqrt(2)]]], dtype=tf.float32)
            self.dwt_high = tf.constant([[[1/np.sqrt(2)]], [[-1/np.sqrt(2)]]], dtype=tf.float32)
            self.dwt_gamma = dwt_gamma
            self.dwt_beta = dwt_beta
            self.dwt_eps = dwt_eps
            self.dwt_levels = dwt_levels

            # historical-magnitude EMA state (f_bar in Eq. 23), one tracker per DWT
            # coefficient group (each level's detail band, plus the final coarsest
            # approximation). train_step and test_step roll out to different
            # lengths, so each needs its own set of trackers.
            train_group_lengths = self._dwt_multilevel_lengths(self.rollout_length, self.dwt_levels)
            vali_group_lengths = self._dwt_multilevel_lengths(self.vali_rollout_length, self.dwt_levels)

            self.fbar_train = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_train_{i}")
                                for i, L in enumerate(train_group_lengths)]
            self.fbar_vali = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_vali_{i}")
                               for i, L in enumerate(vali_group_lengths)]

    # returns number of (trainable) variables/ size of the neural network
    @property
    def trainable_variables(self):
        return self.brain.trainable_variables

    # call to model to predict dependent on the inputs (training set to None)
    def call(self, inputs, training=None):
        return self.brain(inputs, training=training)

    # ---------------- DWT / harmonized-MSE machinery (only used if loss_type="dwt") ----------------
    def _dwt(self, x, length):
        """
        Single-level Haar DWT applied independently to each of the 3 state
        channels (X, Y, Z), via tf.nn.conv1d with the channel dimension folded
        into the batch dimension so channels aren't mixed together.

        x: (batch, length, 3) -> cA, cD: (batch, length//2, 3)
        """
        batch = tf.shape(x)[0]
        coeff_len = length // 2

        x_ch = tf.transpose(x, [0, 2, 1])              # (batch, 3, length)
        x_ch = tf.reshape(x_ch, [-1, length, 1])        # (batch*3, length, 1)

        cA = tf.nn.conv1d(x_ch, self.dwt_low,  stride=2, padding="VALID")  # (batch*3, coeff_len, 1)
        cD = tf.nn.conv1d(x_ch, self.dwt_high, stride=2, padding="VALID")

        cA = tf.transpose(tf.reshape(cA, [batch, 3, coeff_len]), [0, 2, 1])  # (batch, coeff_len, 3)
        cD = tf.transpose(tf.reshape(cD, [batch, 3, coeff_len]), [0, 2, 1])
        return cA, cD

    def _dwt_multilevel(self, x, length, levels):
        """
        Recursive multi-level (Mallat pyramid) Haar DWT, built on the single-level
        _dwt above: at each level, the current approximation is decomposed into a
        new, shorter approximation plus a detail band; the detail band is kept
        and the approximation is fed into the next level.

        x: (batch, length, 3)
        returns: [cD_level1, cD_level2, ..., cD_levelN, cA_levelN]
        """
        coeffs = []
        current, current_length = x, length
        for _ in range(levels):
            cA, cD = self._dwt(current, current_length)
            coeffs.append(cD)
            current, current_length = cA, current_length // 2
        coeffs.append(current)  # final coarse approximation
        return coeffs

    @staticmethod
    def _dwt_multilevel_lengths(length, levels):
        """Coefficient-group lengths for _dwt_multilevel, in the same order it returns them."""
        lengths = []
        current_length = length
        for _ in range(levels):
            current_length = current_length // 2
            lengths.append(current_length)
        lengths.append(current_length)  # final approximation shares the last level's length
        return lengths

    def _harmonized_mse(self, true_seq, pred_seq, length, fbar_list, update):
        """
        Harmonized MSE (Eq. 21) computed on a multi-level Haar DWT of true_seq/pred_seq.

        true_seq, pred_seq - (batch, length, 3) ground-truth / predicted trajectories
        fbar_list           - list of (len_i, 3) EMA trackers, one per DWT coefficient group
        update               - whether to update fbar_list with this batch (Eq. 23)
        """
        true_groups = self._dwt_multilevel(true_seq, length, self.dwt_levels)
        pred_groups = self._dwt_multilevel(pred_seq, length, self.dwt_levels)

        loss = 0.0
        for true_g, pred_g, fbar in zip(true_groups, pred_groups, fbar_list):
            if update:
                batch_mag = tf.reduce_mean(tf.abs(true_g), axis=0)  # (len_i, 3)
                fbar.assign(self.dwt_beta * fbar + (1 - self.dwt_beta) * batch_mag)

            weight = 1.0 + self.dwt_gamma / (fbar + self.dwt_eps)  # (len_i, 3)
            loss += tf.reduce_mean(weight * tf.square(true_g - pred_g))

        return loss

    # heart of the actual training step
    @tf.function
    def train_step(self,data):

        # as you can see here, the training step will recieve data in a tuple of (features,data), which is equivalent to the data you pass to the
        # x= ... and y= .. argument in the keras.Model.fit(...) method
        (features,y) = data

        # features itself contains the X_init values being the values of the state space (X,Y,Z) and the insolation forcing in the input window w at index 0
        # at index 1 the "future" insolation forcing aka. the insolation during the rollout is passed with shape (rollout_length,1)
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        # scale augmentation to randomly rescale state space feature and target trajectories
        scale = tf.random.uniform([batch_size,1,1], 0.8, 1.2)

        # apply scaling to (X,Y,Z) in the input window
        X_init_scaled = tf.concat([X_init[:, :, 0:3] * scale, X_init[:, :, 3:]], axis=-1)
        y_scaled = y * scale

        # tell tensorflow to create the context manager of the variables created in the with statement and tracks them to calculate the gradients
        with tf.GradientTape() as tape:

            # passing input window as first window to use in rollout training
            current_window = X_init_scaled

            # calculate ground truth trajectory, jacobian and hessian
            true_path = tf.concat([X_init_scaled[:, -1:, 0:3], y_scaled], axis=1)

            true_jacobian = true_path[:, 1:, :] - true_path[:, :-1, :]
            true_hessian = true_jacobian[:, 1:, :] - true_jacobian[:, :-1, :]

            all_paths= []
            all_jacobian = []

            # actual rollout training
            for i in range(self.rollout_length):

                # current window (first step this is just the input window) with added normally distributed (input) noise
                noisy_window = current_window + tf.random.normal(tf.shape(current_window), stddev=self.noise_level)

                # generating increment predictions (forward-pass)
                d_prediction = self.brain(noisy_window, training=True)

                # adding predicted increment to last entry in the crurrent windoe
                x_now = current_window[:, -1, 0:3]
                prediction_abs = x_now + d_prediction

                all_paths.append(prediction_abs)
                all_jacobian.append(d_prediction)

                # adding additional output noise
                prediction_abs += tf.random.normal(tf.shape(prediction_abs), stddev=self.noise_level)

                next_step = tf.zeros((batch_size, 1, n_features))

                pred = tf.expand_dims(prediction_abs, axis=1)

                forc_true = future_forcing[:, i]

                # noise injection usedto prevent NN from remembering just the values, resulting in insolation just being used as a clock (safety measure)
                forc_noisy = forc_true + tf.random.normal(tf.shape(forc_true), stddev=self.noise_level)

                forc = tf.reshape(forc_noisy, (batch_size, 1, 1))
                next_step = tf.concat([pred,forc],axis=-1)

                # appending next predicted state and insolation forcig to the current window and moving window one time step forward
                current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

            # creating stacks over rollout window for trajectory and jacobian (used
            # by "sobolev" mode below; harmless extra bookkeeping in "dwt" mode)
            p_stack = tf.stack(all_paths, axis=1)
            jacobian_stack = tf.stack(all_jacobian, axis=1)
            pred_hessian = jacobian_stack[:, 1:, :] - jacobian_stack[:, :-1, :]

            if self.loss_type == "dwt":
                # Harmonized MSE on a multi-level Haar DWT of the trajectory (Eq. 21)
                abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.rollout_length,
                                                 self.fbar_train, update=True)
                total_loss = self.L_2_weight * abs_loss
                metrics = {"abs": abs_loss}
            else:
                # Sobolev loss: pointwise MSE + jacobian + hessian
                abs_loss = tf.reduce_mean(tf.square(true_path[:,1:,:]- p_stack))
                jacobian_loss = tf.reduce_mean(tf.square(true_jacobian - jacobian_stack))
                hessian_loss = tf.reduce_mean(tf.square(true_hessian - pred_hessian))
                total_loss = (self.L_2_weight * abs_loss) + (self.jacobian_weight * jacobian_loss) + (self.hessian_weight * hessian_loss)
                metrics = {"abs": abs_loss, "jacobian": jacobian_loss, "hessian": hessian_loss}

        # Backpropagation through the forward pass operations tracked by GradientTape()
        grads = tape.gradient(total_loss,self.trainable_variables)
        # weight update
        self.optimizer.apply_gradients(zip(grads,self.trainable_variables))

        return {"loss": total_loss, **metrics}

    # test_step used in the validation process
    # essentially the same as the training step, only with training disabled and no backpropagation/ weights update
    def test_step(self,data):

        def calculate_lag1_acf(p_stack):
            """
            calculate the lag_1 auto-correlation function to ensure causality in the validation step is preserved
            --------------------------------------------------

            p_stack - predicted validation rollout trajetory
            """

            mu = tf.reduce_mean(p_stack, axis=1, keepdims=True)
            var = tf.math.reduce_variance(p_stack, axis=1, keepdims=True)

            p_centered = p_stack - mu
            t0 = p_centered[:, :-1, :]
            t1 = p_centered[:, 1:, :]

            acf_per_sample = tf.reduce_mean(t0 * t1, axis=1, keepdims=True) / (var + 1e-7)

            return tf.reduce_mean(acf_per_sample)


        (features,y) = data
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        current_window = X_init

        all_paths= []
        all_jacobian = []

        true_path = tf.concat([X_init[:, -1:, 0:3], y], axis=1) # [Batch, Rollout+1, 3]
        true_jacobian = true_path[:, 1:, :] - true_path[:, :-1, :] # [Batch, Rollout, 3]
        true_hessian = true_jacobian[:, 1:, :] - true_jacobian[:, :-1, :]


        for i in range(self.vali_rollout_length):
            d_prediction = self.brain(current_window, training=False)

            x_now = current_window[:, -1, 0:3]
            prediction_abs = x_now + d_prediction

            all_paths.append(prediction_abs)
            all_jacobian.append(d_prediction)

            next_step = tf.zeros((batch_size, 1, n_features))

            pred = tf.expand_dims(prediction_abs, axis=1)
            forc = tf.reshape(future_forcing[:, i], (batch_size, 1, 1))
            next_step = tf.concat([pred,forc],axis=-1)

            current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

        p_stack = tf.stack(all_paths, axis=1)

        jacobian_stack = tf.stack(all_jacobian, axis=1)
        pred_hessian = jacobian_stack[:, 1:, :] - jacobian_stack[:, :-1, :]

        if self.loss_type == "dwt":
            # own EMA tracker, since the coefficient count differs from training's rollout length
            abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.vali_rollout_length,
                                             self.fbar_vali, update=True)
            total_loss = self.L_2_weight * abs_loss
            metrics = {"abs": abs_loss}
        else:
            abs_loss = tf.reduce_mean(tf.square(true_path[:,1:,:]- p_stack))
            jacobian_loss = tf.reduce_mean(tf.square(true_jacobian - jacobian_stack))
            hessian_loss = tf.reduce_mean(tf.square(true_hessian - pred_hessian))
            total_loss = (self.L_2_weight * abs_loss) + (self.jacobian_weight * jacobian_loss) + (self.hessian_weight * hessian_loss)
            metrics = {"abs": abs_loss, "jacobian": jacobian_loss, "hessian": hessian_loss}

        lag1_acf = calculate_lag1_acf(p_stack)

        # calculate standard deviation for ground truth and prediction in validation step
        y_std = tf.math.reduce_std(y,axis = 1)
        p_std = tf.math.reduce_std(p_stack, axis=1)

        # calculate C_2 constraint
        C_2 =  tf.reduce_mean(tf.abs(1.0 - (p_std / (y_std + 1e-6))))

        return {"loss": total_loss, **metrics, "lag1_acf": lag1_acf, "C_2": C_2}

class PhysicalAttractorCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self,filepath="best_state_space_model.weights.h5",acf_threshold=0.97):
        """
        initializer method for C_2 constraint callback
        ------------------------------------------------

        filepath               - file path to save the best model weights
        acf_threshold          - threshold fro acf_function to accept model
        """
        super(PhysicalAttractorCheckpoint,self).__init__()
        self.filepath = filepath
        self.acf_threshold = acf_threshold
        self.best_C_2 = float('inf')
        self.best_epoch = 0
        self.patience = 0

    def on_epoch_end(self,epoch=0,logs=None):

        # retrieve current C_2 and lag_1 ACF values
        current_C_2 = logs.get("val_C_2")
        current_acf = logs.get("val_lag1_acf")

        if current_C_2 is None or current_acf is None:
            print("\nWarning: Physical metrics not found in logs. Check test_step return keys.")
            return

        # convert from string to float
        current_C_2 = float(current_C_2)
        current_acf = float(current_acf)

        # check if ACF criteria is meet
        if current_acf < self.acf_threshold:
            if (epoch + 1) % 5 == 0:
                print(f"\nEpoch {epoch+1}: Smoothness ({current_acf:.4f}) below threshold. Ignoring.")
            return

        # check if current model returns a lower C_2 constraint then previous model, if yes overwrite the current best model
        if current_C_2 < self.best_C_2:
            self.best_C_2 = current_C_2
            self.best_epoch = epoch
            self.model.save_weights(self.filepath)
            self.patience = 0
            print(f"\nEpoch {epoch+1}: NEW PHYSICAL BEST!")
            print(f" > C_2: {current_C_2:.4f}")
            print(f" > Lag-1 ACF:    {current_acf:.4f}")
            print(f" > Weights saved to {self.filepath}")
        else:
            self.patience += 1
            if self.patience == 20:
                self.model.stop_training = True
                print(f"\nPatience ran out. Early stopping at epoch {epoch}: no physically-valid "
                      f"improvement in val_C_2 for {self.patience} epochs.")


## Load and set up data

In [ ]:
# set random initilization seed for
set_seeds()
plt.rcParams.update({'font.size': 20})

timestep = 1 #kyr
# define input window size, training and validation rollout length
window_size = int(50/timestep)# dividing by timestep to get number of time steps in 50kyr window
rollout_length = int(30/timestep) # " " for 20kyr rollout length 
rollout_length_validation = int(100/timestep)

mappings = ["X","Y","Z"]

color_ls = [["#D6195E","#FFB600"],["#1E88E5","#6519C5"],["#05EFC5","#8DB1A7"]]


In [ ]:
# import interpolated observational data (dO18, CO2, MgCa, insolation vs. age)
# attempt 1 uses LR04-based data + EDC CO2 to train; uses LR04-based data to test
obs_data = pd.read_csv(
    f'{PROJECT}/Data/InterpolatedObs/gpr_full_traj_2diff.csv'
).iloc[::-1].reset_index(drop=True)

# numpy arrays for each state variable, the insolation forcing, and the shared time axis
X = obs_data['dO18_LR04[pm]'].to_numpy(dtype=float)
Y = obs_data['CO2_Yamamoto[pm]'].to_numpy(dtype=float)
Z = obs_data['MgCa_Eld[mmol/mol]'].to_numpy(dtype=float)
R = obs_data['Insolation[W/m2]'].to_numpy(dtype=float)
time = obs_data['Age[kaBP]'].to_numpy(dtype=float)

print(f"data shape: {obs_data.shape}")

obs_data.head()


In [ ]:
# fraction of the data held out as the test set (oldest chunk), and the
# train:validation split of the remaining train/validation set
test_frac = 0.2
train_frac = 0.8

# split data into a test set (oldest ~test_frac of the record) and a train/validation
# set (the remaining, more recent portion), with a 1-row buffer between them so the
# test seed window can't straddle the split
n_total = X.shape[0]
test_size = int(round(test_frac * n_total))

test_data = np.column_stack([X[:test_size], Y[:test_size], Z[:test_size], R[:test_size]])
test_time = time[:test_size]

trainval = np.column_stack([X[test_size+1:], Y[test_size+1:], Z[test_size+1:], R[test_size+1:]])
trainval_time = time[test_size+1:]

fulldata = np.column_stack([X, Y, Z, R])
fulldata_time = time

# train:validation split of the train/validation set
n = trainval.shape[0]
train_end = int(round(train_frac * n))

train_data = trainval[:train_end, :]
validation_data = trainval[train_end:, :]

print(f"test: {test_data.shape} (ages {test_time.min():.0f}-{test_time.max():.0f}), "
      f"trainval: {trainval.shape} (ages {trainval_time.min():.0f}-{trainval_time.max():.0f}), "
      f"train: {train_data.shape}, validation: {validation_data.shape}")


In [ ]:
# fit standard scaler to train/valid data
train_scaler = StandardScaler()
train_scaler.fit(train_data)
# fit standard scaler to test data

# scale data with scaler
obs_train_data = train_scaler.transform(train_data)
test_data = train_scaler.transform(test_data)
obs_valid_data = train_scaler.transform(validation_data)

X_train_obs, y_train_obs, Insol_train_obs = create_windows(obs_train_data, window_size, rollout_length)
X_valid_obs, y_valid_obs, Insol_valid_obs = create_windows(obs_valid_data, window_size, rollout_length_validation)



## Train NN

In [ ]:

def objective(trial, loss_type="sobolev", verbose = False, obs_train_data = obs_train_data, obs_valid_data = obs_valid_data):
    # loss_type: "sobolev" (pointwise MSE + jacobian + hessian) or "dwt" (harmonized
    # MSE on a multi-level Haar DWT of the rollout) - see RolloutEmulator above.
    # This is a plain function argument rather than something Optuna searches over,
    # since the two losses aren't on a comparable scale - mixing them in one study
    # would make trial comparisons meaningless. To run one architecture vs. the
    # other, call e.g. study.optimize(lambda trial: objective(trial, loss_type="dwt"), ...)

    convolutional_filters = trial.suggest_int('conv filters', 10, 32)
    convolutional_kernels = trial.suggest_int('conv kernel', 3, 10)
    dense_units = trial.suggest_int('dense unit', 10, 40)
    rate = trial.suggest_float('dropout rate', 0.1, 0.4)
    learning_rate = trial.suggest_float('learning rate', 1e-5, 1e-3)

    if loss_type == 'dwt':
        rollout_length = 32
        # rollout length not included as trainable parameter bc fixed to 32 (to ensure divisibility by 2^dwt_levels)
        dwt_gamma = trial.suggest_float('dwt gamma', 0.1, 1.0)
        dwt_beta = trial.suggest_float('dwt beta', 0.1, 0.9)
        dwt_eps = trial.suggest_float('dwt eps', 1e-8, 1e-4, log=True)
        dwt_levels = trial.suggest_int('dwt levels', 2, 4)
    else:
        rollout_length = trial.suggest_int('training rollout length', 10, 35)



    window_size = int(50)
    rollout_length_validation = int(100)

    X_train_obs, y_train_obs, Insol_train_obs = create_windows(obs_train_data, window_size, rollout_length)
    X_valid_obs, y_valid_obs, Insol_valid_obs = create_windows(obs_valid_data, window_size, rollout_length_validation)

    np.save("train_set",train_data[:,:-1])
    np.save("valid_set",validation_data[:,:-1])

    brain = CNN_model(window_size=window_size, 
                      n_features=4, 
                      conv_filter = convolutional_filters, 
                      conv_kernel = convolutional_kernels, 
                      dense_unit = dense_units,
                      rate = rate)
    if verbose:
        brain.summary()

    if loss_type == "dwt":
        model = RolloutEmulator(
            brain,
            rollout_length=rollout_length,
            vali_rollout_length=rollout_length_validation,
            window_size=window_size,
            n_features=4,
            loss_type=loss_type,
            dwt_gamma=dwt_gamma,
            dwt_beta=dwt_beta,
            dwt_eps=dwt_eps,
            dwt_levels=dwt_levels,
        )
    else:
        model = RolloutEmulator(
        brain,
        rollout_length=rollout_length,
        vali_rollout_length=rollout_length_validation,
        window_size=window_size,
        n_features=4,
        loss_type=loss_type,
        )

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0))

    dummy_X = tf.zeros((1, window_size, 4))
    _ = model(dummy_X)

    phys_check = PhysicalAttractorCheckpoint(
        filepath="best_state_space_model.weights.h5",
        acf_threshold=0.98,
    )

    history = model.fit(
        [X_train_obs, Insol_train_obs],
        y_train_obs,
        validation_data=([X_valid_obs, Insol_valid_obs], y_valid_obs),
        epochs=100,
        callbacks=[phys_check],
        batch_size=64,
        validation_batch_size=64,
        shuffle=True,
    )

    #best_c2 = phys_check.best_C_2
    patience = 20
    objective_score = np.mean(history.history['val_C_2'][-(patience+1):])

    return objective_score



In [ ]:
study = opt.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, loss_type="dwt"), n_trials=50)


In [ ]:
study.best_trials


In [ ]:
'''
OPTIMIZATION #5
[FrozenTrial(number=39, state=<TrialState.COMPLETE: 1>, values=[0.19121956044719332], datetime_start=datetime.datetime(2026, 9, 22, 14, 56, 58, 296974), datetime_complete=datetime.datetime(2026, 9, 22, 14, 58, 19, 831112), 
params={'conv filters': 16, 'conv kernel': 9, 'dense unit': 21, 'dropout rate': 0.24199127728779032, 'learning rate': 0.0008599275973935869, 'dwt gamma': 0.4016606625962723, 'dwt beta': 0.4538313866456523, 'dwt eps': 1.5028933889868065e-06, 'dwt levels': 2}, 
user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1), 'dropout rate': FloatDistribution(high=0.4, log=False, low=0.1, step=None), 'learning rate': FloatDistribution(high=0.001, log=False, low=1e-05, step=None), 'dwt gamma': FloatDistribution(high=1.0, log=False, low=0.1, step=None), 'dwt beta': FloatDistribution(high=0.9, log=False, low=0.1, step=None), 'dwt eps': FloatDistribution(high=0.0001, log=True, low=1e-08, step=None), 'dwt levels': IntDistribution(high=4, log=False, low=2, step=1)}, trial_id=39, value=None)]
'''
'''
OPTIMIZATION #4
[FrozenTrial(number=14, state=<TrialState.COMPLETE: 1>, values=[0.08588084081808726], datetime_start=datetime.datetime(2026, 9, 2, 16, 27, 50, 115970), datetime_complete=datetime.datetime(2026, 9, 2, 16, 29, 27, 991276), 
params={'conv filters': 32, 'conv kernel': 9, 'dense unit': 10, 'dropout rate': 0.2632450164100955, 'learning rate': 0.0005192120606179918, 'dwt gamma': 0.3818453319933549, 'dwt beta': 0.21402907131795812, 'dwt eps': 3.61758437807838e-08, 'dwt levels': 3}, 
user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1), 'dropout rate': FloatDistribution(high=0.4, log=False, low=0.1, step=None), 'learning rate': FloatDistribution(high=0.001, log=False, low=0.0001, step=None), 'dwt gamma': FloatDistribution(high=1.0, log=False, low=0.1, step=None), 'dwt beta': FloatDistribution(high=0.9, log=False, low=0.1, step=None), 'dwt eps': FloatDistribution(high=0.0001, log=True, low=1e-08, step=None), 'dwt levels': IntDistribution(high=3, log=False, low=2, step=1)}, trial_id=14, value=None)]
'''
'''
OPTIMIZATION #3
[FrozenTrial(number=35, state=<TrialState.COMPLETE: 1>, values=[0.14120963570617495], datetime_start=datetime.datetime(2026, 8, 24, 17, 40, 19, 482224), datetime_complete=datetime.datetime(2026, 8, 24, 17, 41, 10, 859554), params={'training rollout length': 32, 'conv filters': 18, 'conv kernel': 10, 'dense unit': 20, 'dropout rate': 0.18993441674411538, 'learning rate': 0.0014870705892306704}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1), 'dropout rate': FloatDistribution(high=0.5, log=False, low=0.05, step=None), 'learning rate': FloatDistribution(high=0.01, log=False, low=0.0001, step=None)}, trial_id=35, value=None)]

'''
'''
OPTIMIZATION #2
[FrozenTrial(number=20, state=<TrialState.COMPLETE: 1>, values=[0.08312221616506577], datetime_start=datetime.datetime(2026, 8, 24, 15, 14, 48, 218548), datetime_complete=datetime.datetime(2026, 8, 24, 15, 15, 42, 764783), params={'training rollout length': 28, 'conv filters': 18, 'conv kernel': 10, 'dense unit': 17, 'dropout rate': 0.0602137356190553, 'learning rate': 0.006494894059024996}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1), 'dropout rate': FloatDistribution(high=0.5, log=False, low=0.05, step=None), 'learning rate': FloatDistribution(high=0.01, log=False, low=0.0001, step=None)}, trial_id=20, value=None)]
'''
'''
OPTIMIZATION #1
FrozenTrial(number=8, state=<TrialState.COMPLETE: 1>, values=[0.10665134340524673, 0.10664883255958557], datetime_start=datetime.datetime(2026, 8, 21, 17, 55, 40, 33758), datetime_complete=datetime.datetime(2026, 8, 21, 17, 57, 45, 313919), params={'training rollout length': 34, 'conv filters': 26, 'conv kernel': 8, 'dense unit': 16}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=16, state=<TrialState.COMPLETE: 1>, values=[0.07314169406890869, 1.121516540646553], datetime_start=datetime.datetime(2026, 8, 21, 18, 7, 38, 702947), datetime_complete=datetime.datetime(2026, 8, 21, 18, 9, 51, 181410), params={'training rollout length': 33, 'conv filters': 26, 'conv kernel': 9, 'dense unit': 22}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=24, state=<TrialState.COMPLETE: 1>, values=[0.09361737966537476, 0.8256790041923523], datetime_start=datetime.datetime(2026, 8, 21, 18, 19, 7, 815356), datetime_complete=datetime.datetime(2026, 8, 21, 18, 20, 26, 702148), params={'training rollout length': 28, 'conv filters': 12, 'conv kernel': 3, 'dense unit': 28}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=40, state=<TrialState.COMPLETE: 1>, values=[0.09555989503860474, 0.5789969861507416], datetime_start=datetime.datetime(2026, 8, 21, 18, 40, 6, 771168), datetime_complete=datetime.datetime(2026, 8, 21, 18, 41, 34, 658457), params={'training rollout length': 17, 'conv filters': 31, 'conv kernel': 10, 'dense unit': 39}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=47, state=<TrialState.COMPLETE: 1>, values=[0.15873461961746216, 0.10536447167396545], datetime_start=datetime.datetime(2026, 8, 21, 18, 49, 58, 794450), datetime_complete=datetime.datetime(2026, 8, 21, 18, 51, 24, 714728), params={'training rollout length': 25, 'conv filters': 15, 'conv kernel': 9, 'dense unit': 27}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'training rollout length': IntDistribution(high=35, log=False, low=10, step=1), 'conv filters': IntDistribution(high=32, log=False, low=10, step=1), 'conv kernel': IntDistribution(high=10, log=False, low=3, step=1), 'dense unit': IntDistribution(high=40, log=False, low=10, step=1)}, trial_id=47, value=None)]
'''